# Using AutoTokenizer and AutoModel to Get Embeddings

In [1]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F
from scipy.stats import spearmanr
import numpy as np

In [2]:
# model local path
local_path = "/Users/sir/Downloads/HuggingFace/sentence_transformer/"
# Global device and model initialization for performance
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(local_path + "all-mpnet-base-v2")
model = AutoModel.from_pretrained(local_path + "all-mpnet-base-v2").to(DEVICE)

In [3]:

print(model)

MPNetModel(
  (embeddings): MPNetEmbeddings(
    (word_embeddings): Embedding(30527, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): MPNetEncoder(
    (layer): ModuleList(
      (0-11): 12 x MPNetLayer(
        (attention): MPNetAttention(
          (attn): MPNetSelfAttention(
            (q): Linear(in_features=768, out_features=768, bias=True)
            (k): Linear(in_features=768, out_features=768, bias=True)
            (v): Linear(in_features=768, out_features=768, bias=True)
            (o): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (intermediate): MPNetIntermediate(
          (dense): Linear(in_

In [4]:
def mean_pooling_with_mask(model_output, attention_mask):
    # model_output[0] is the last_hidden_state (batch_size, sequence_length, hidden_size)
    token_embeddings = model_output.last_hidden_state
    
    # 1. Expand attention mask to match the embedding dimension
    # (batch_size, sequence_length, 1) -> (batch_size, sequence_length, hidden_size)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    
    # 2. Sum the real tokens (zeros out padding tokens)
    sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
    
    # 3. Get the number of real tokens per sentence (must be at least 1 to avoid division by zero)
    sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    # 4. Divide the sum by the number of real tokens to get the average
    return sum_embeddings / sum_mask


def embed_texts(texts: list[str]) -> np.ndarray:
    """
    Generates embeddings for a list of texts in an optimized, batch-aware manner.

    Optimizations:
    1. Processes a batch (list) of texts at once.
    2. Uses torch.float16 (half-precision) for faster GPU calculation.
    3. Manages tensors on the detected DEVICE (CUDA or CPU).

    Args:
        texts: A list of strings to embed.

    Returns:
        A NumPy array where each row is the embedding vector for a corresponding text.
    """
    if not tokenizer or not model:
        raise RuntimeError("Model and tokenizer must be initialized before calling embed_texts.")

    # 1. Tokenization and Device Transfer (Batch Processing)
    inputs = tokenizer(
        texts, 
        return_tensors="pt", 
        truncation=True, 
        padding=True,
        max_length=512 # Good practice to specify max length
    ).to(DEVICE)
    
    # 2. Model Inference
    with torch.no_grad():
        outputs = model(**inputs)

    # 3. Mean Pooling and Conversion to NumPy
    # Use mean pooling of the last hidden state over the sequence length dimension (dim=1)
    embeddings = mean_pooling_with_mask(outputs, inputs['attention_mask'])
    # embeddings = outputs.last_hidden_state.mean(dim=1)

    # 4. L2 NORMALIZATION
    # Normalize the vectors to have a length of 1 along the feature dimension (dim=1)
    normalized_embeddings = F.normalize(embeddings, p=2, dim=1)
    
    # 5. Conversion to NumPy
    return normalized_embeddings.cpu().numpy()


In [5]:
# less than 512 tokens per sentence
sentences1 = ["I love dogs", "He is driving a car", "AI is amazing", "The sentence transformers are great!", "you are dead.", "The dog is fast."]
sentences2 = ["I adore dogs", "He drives a vehicle", "Artificial Intelligence is great", "The Large Language mode1s are fantastic.", "you are alive.", 
              "The dog is Not fast."]

In [6]:
# lets get embeddings
one = embed_texts(sentences1)
two = embed_texts(sentences2)

print(f"Embedding shape: {one.shape} & Model Contex length: {model.config.max_position_embeddings}")
print(f"Embedding shape: {two.shape} & Tokenizer Contex length: {tokenizer.model_max_length}") 

Embedding shape: (6, 768) & Model Contex length: 514
Embedding shape: (6, 768) & Tokenizer Contex length: 384


### Understanding the Context Length Discrepancy

1. Model Contex length: `514` This value, accessed via model.config.max_position_embeddings, is the architectural limit of the underlying MPNet model.It tells you the size of the model's positional embedding matrix. The model can technically process a sequence up to `514` tokens long (including the `[CLS]` and `[SEP]` special tokens) without an out-of-bounds error.The value is `514` instead of the more common `512` because the original MPNet model was designed slightly differently from the standard BERT/RoBERTa architectures.
2. Tokenizer Contex length: `384` This value, accessed via tokenizer.model_max_length, is the recommended/default operational limit for the specific pre-trained task (in this case, sentence embedding).The all-mpnet-base-v2 model was trained by Sentence Transformers using a maximum sequence length of `384` tokens. Crucially, tokens beyond `384` were truncated during the training of this specific sentence-embedding model. Using a sequence longer than `384` means you are feeding tokens into positional embeddings that were not properly learned during the fine-tuning process, which can drastically reduce the quality of your embeddings.

**Note:**
One should always trust and adhere to the smaller value that comes from the fine-tuned configuration, which is the Tokenizer Context Length: 384.

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

# Compute cosine similarity between embeddings
similarity = cosine_similarity(one, two)

similarity.diagonal()

array([0.94654566, 0.76061594, 0.7782569 , 0.21364242, 0.63889325,
       0.93956894], dtype=float32)

# Dot product between two embeddings
- we should get same values

In [8]:
# Compute row norms
one_norms = np.linalg.norm(one, axis=1, keepdims=True)
two_norms = np.linalg.norm(two, axis=1, keepdims=True)

# Normalize rows in-place by dividing each element by the row norm
one /= one_norms
two /= two_norms

cosine_similarities = np.sum(one * two, axis=1)
cosine_similarities = np.dot(one,two.T).diagonal()

cosine_similarities

array([0.94654566, 0.76061594, 0.7782569 , 0.21364242, 0.63889325,
       0.93956894], dtype=float32)

# Different Model

In [9]:
tokenizer = AutoTokenizer.from_pretrained(local_path + "nli-mpnet-base-v2")
model = AutoModel.from_pretrained(local_path + "nli-mpnet-base-v2").to(DEVICE)

# lets get embeddings
one = embed_texts(sentences1)
two = embed_texts(sentences2)

# Compute row norms
one_norms = np.linalg.norm(one, axis=1, keepdims=True)
two_norms = np.linalg.norm(two, axis=1, keepdims=True)

# Normalize rows in-place by dividing each element by the row norm
one /= one_norms
two /= two_norms

cosine_similarities = np.sum(one * two, axis=1)
cosine_similarities = np.dot(one,two.T).diagonal()

print(cosine_similarities)
print(f"\nEmbedding shape: {one.shape} & Model Contex length: {model.config.max_position_embeddings}")
print(f"Embedding shape: {two.shape} & Tokenizer Contex length: {tokenizer.model_max_length}") 

[0.9236439  0.81375355 0.45824954 0.27817354 0.35801485 0.81164294]

Embedding shape: (6, 768) & Model Contex length: 514
Embedding shape: (6, 768) & Tokenizer Contex length: 75


In [10]:
# Using PyTorch for cosine similarity
one_tensor = torch.tensor(one).to(DEVICE)
two_tensor = torch.tensor(two).to(DEVICE)


cosine_similarities = F.cosine_similarity(one_tensor, two_tensor).to('cpu').numpy()
cosine_similarities

array([0.9236438 , 0.8137537 , 0.4582495 , 0.2781734 , 0.35801512,
       0.8116431 ], dtype=float32)

Spearman’s rank correlation coefficient (denoted as $\rho$ or $r_s$) is often used to evaluate the agreement between cosine similarity scores from vector embeddings and human-labeled similarity scores.

In [11]:
from scipy.stats import spearmanr

# Example cosine similarity scores and human-labeled scores
human_scores = [5, 4, 4, 2 ,0, 0] 

correlation, p_value = spearmanr(cosine_similarities, human_scores)
print(f"Spearman's rank correlation: {correlation} & {p_value}")

Spearman's rank correlation: 0.6179143806533246 & 0.1910939016901931


#### Cosine similarity can range from -1 to 1, but the human scores are 0–5. We can scale the cosine scores to the same range:

In [12]:
from sklearn.preprocessing import MinMaxScaler

# Convert cosine similarities to a NumPy array
cosine_similarities = np.array(cosine_similarities).reshape(-1, 1)

# Scale to range [0, 5] to match human scores
scaler = MinMaxScaler(feature_range=(0, 5))
cosine_scaled = scaler.fit_transform(cosine_similarities).flatten()

print(f"Scaled cosine similarities: {cosine_scaled}")

correlation, p_value = spearmanr(cosine_scaled, human_scores)
print(f"\nSpearman's rank correlation after scaling: {correlation:.4f} & p-value: {p_value:.4f}")

Scaled cosine similarities: [5.        4.14876   1.3949218 0.        0.6184771 4.13241  ]

Spearman's rank correlation after scaling: 0.6179 & p-value: 0.1911


#### Sometimes, small differences in embeddings dominate the raw cosine similarity. You can compress extreme values to reduce noise:

In [13]:
cosine_smoothed = 5 / (1 + np.exp(-10 * (cosine_scaled/5 - 0.5)))  # Sigmoid to [0,5]
correlation_smoothed, p_value = spearmanr(cosine_smoothed, human_scores)
print(f"Spearman's rank correlation after sigmoid smoothing: {correlation_smoothed:.4f} & p-value: {p_value:.4f}")

Spearman's rank correlation after sigmoid smoothing: 0.6179 & p-value: 0.1911


# Smaller embedding Model (all-MiniLM-L6-v2)

In [14]:
tokenizer = AutoTokenizer.from_pretrained(local_path + "all-MiniLM-L6-v2")
model = AutoModel.from_pretrained(local_path + "all-MiniLM-L6-v2").to(DEVICE)

In [15]:
# lets get embeddings
one = embed_texts(sentences1)
two = embed_texts(sentences2)

print(f"\nEmbedding shape: {one.shape} & Model Contex length: {model.config.max_position_embeddings}")
print(f"Embedding shape: {two.shape} & Tokenizer Contex length: {tokenizer.model_max_length}") 


Embedding shape: (6, 384) & Model Contex length: 512
Embedding shape: (6, 384) & Tokenizer Contex length: 256


In [16]:
# Using PyTorch for cosine similarity
one_tensor = torch.tensor(one).to(DEVICE)
two_tensor = torch.tensor(two).to(DEVICE)


cosine_similarities = F.cosine_similarity(one_tensor, two_tensor).to('cpu').numpy()
cosine_similarities

array([0.83271223, 0.82757056, 0.73890233, 0.33188027, 0.80207694,
       0.96619827], dtype=float32)

In [17]:
# Example cosine similarity scores and human-labeled scores
human_scores = [5, 4, 4, 2 ,0, 0] 

correlation, p_value = spearmanr(cosine_similarities, human_scores)
print(f"Spearman's rank correlation: {correlation} & {p_value}")

Spearman's rank correlation: 0.0 & 1.0


In [18]:
from sentence_transformers import SentenceTransformer, util

# Input sentences
s1 = "The dog is fast."
s2 = "The dog is not fast."

# Load model and move to DEVICE
model = SentenceTransformer(local_path + "nli-mpnet-base-v2").to(DEVICE)


# Encode and calculate
embeddings1 = model.encode([s1], convert_to_tensor=True).to(DEVICE)
embeddings2 = model.encode([s2], convert_to_tensor=True).to(DEVICE)
similarity = util.cos_sim(embeddings1, embeddings2).item()

print(f"--- Model:nli-mpnet-base-v2'---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Actual Cosine Similarity: {similarity:.8f}")
print(f"Embedding shape: {embeddings2.shape} & Tokenizer Contex length: {model.max_seq_length}")

--- Model:nli-mpnet-base-v2'---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Actual Cosine Similarity: 0.81164283
Embedding shape: torch.Size([1, 768]) & Tokenizer Contex length: 75


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [19]:
from sentence_transformers import SentenceTransformer, util

# Input sentences
s1 = "The dog is fast."
s2 = "The dog is not fast."

# Load model and move to DEVICE
model = SentenceTransformer(local_path + "all-MiniLM-L6-v2").to(DEVICE)

# Encode and calculate
embeddings1 = model.encode([s1], convert_to_tensor=True).to(DEVICE)
embeddings2 = model.encode([s2], convert_to_tensor=True).to(DEVICE)
similarity = util.cos_sim(embeddings1, embeddings2).item()

print(f"--- Model:all-MiniLM-L6-v2'---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Actual Cosine Similarity: {similarity:.8f}")
print(f"Embedding shape: {embeddings1.shape} & Tokenizer Contex length: {model.max_seq_length}")

--- Model:all-MiniLM-L6-v2'---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Actual Cosine Similarity: 0.96619821
Embedding shape: torch.Size([1, 384]) & Tokenizer Contex length: 256


In [20]:
from sentence_transformers import CrossEncoder


local_path = "/Users/sir/Downloads/HuggingFace/cross_encoder/"

# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder(local_path + "cross-encoder_stsb-roberta-base").to(DEVICE)

# Predict similarity score
similarity = model.predict([(s1, s2)])[0]

print(f"--- Model: cross-encoder/stsb-roberta-base ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Tokenizer Contex length: {model.max_length} & Predicted Similarity Score: {similarity:.4f}")

--- Model: cross-encoder/stsb-roberta-base ---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Tokenizer Contex length: 512 & Predicted Similarity Score: 0.2924


In [21]:
# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder(local_path + 'cross-encoder_nli-distilroberta-base').to(DEVICE)

# Predict logits (1D tensor)
logits = model.predict([(s1, s2)], convert_to_tensor=True)  # shape: [3]

# Apply softmax along dim=0
probs = F.softmax(logits, dim=1)  # still a 1D tensor

labels = ["entailment", "neutral", "contradiction"]

# Iterate correctly: convert to list of floats
for label, prob in zip(labels, probs[0]):
    print(f"{label:15s}: {prob.item():.4f}")


print(f"Tokenizer Contex length: {model.max_length}")

entailment     : 0.9907
neutral        : 0.0050
contradiction  : 0.0043
Tokenizer Contex length: 512


In [22]:
# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder(local_path + 'cross-encoder_nli-deberta-v3-small').to(DEVICE)

# Predict logits (1D tensor)
logits = model.predict([(s1, s2)], convert_to_tensor=True)  # shape: [3]

# Apply softmax along dim=0
probs = F.softmax(logits, dim=1)  # still a 1D tensor

labels = ["entailment", "neutral", "contradiction"]

# Iterate correctly: convert to list of floats
for label, prob in zip(labels, probs[0]):
    print(f"{label:15s}: {prob.item():.4f}")

print(f"Tokenizer Contex length: {model.max_length}")

entailment     : 0.9965
neutral        : 0.0012
contradiction  : 0.0023
Tokenizer Contex length: 512


In [23]:
# Load a cross-encoder that directly outputs similarity scores
model = CrossEncoder(local_path + 'cross-encoder_stsb-roberta-large').to(DEVICE)

# Predict similarity score
similarity = model.predict([(s1, s2)])

print(f"--- Model: cross-encoder/stsb-roberta-large ---")
print(f"Sentence 1: '{s1}'")
print(f"Sentence 2: '{s2}'")
print(f"Tokenizer Contex length: {model.max_length} & Predicted Similarity Score: {similarity[0]:.6f}")

--- Model: cross-encoder/stsb-roberta-large ---
Sentence 1: 'The dog is fast.'
Sentence 2: 'The dog is not fast.'
Tokenizer Contex length: 512 & Predicted Similarity Score: 0.463631


In [24]:
# Define sentences
s1 = "The dog is very fast."
s2 = "The dog is not fast."

In [25]:
# Use a stronger NLI model trained on negation-sensitive data
model = CrossEncoder(local_path + 'cross-encoder_nli-deberta-v3-small').to(DEVICE)

# Predict logits (3 values: entailment, neutral, contradiction)
logits = model.predict([(s1, s2)], convert_to_tensor=True)

# Apply softmax to get probabilities
probs = F.softmax(logits, dim=1)

# Map to labels
labels = ["entailment", "neutral", "contradiction"]

# Show results
for label, prob in zip(labels, probs[0]):
    print(f"{label:15s}: {prob.item():.6f}")


print(f"Tokenizer Contex length: {model.max_length}")

entailment     : 0.997232
neutral        : 0.001059
contradiction  : 0.001709
Tokenizer Contex length: 512


In [26]:
# Initialize the CrossEncoder model
model = CrossEncoder(local_path + 'cross-encoder_nli-deberta-v3-base').to(DEVICE)

# Sentence pairs for NLI
sentence_pairs = [
    ("The dog is very fast."),
    ("The dog is not fast.")
]

# Predict logits for each pair
logits = model.predict(sentence_pairs)

# Apply softmax to get probabilities
probs = F.softmax(torch.from_numpy(logits), dim=0)

# Map to labels
labels = ["entailment", "neutral", "contradiction"]

# Show results
for label, prob in zip(labels, probs):
    print(f"{label:15s}: {prob.item():.6f}")

print(f"Tokenizer Contex length: {model.max_length}")

entailment     : 0.999285
neutral        : 0.000220
contradiction  : 0.000495
Tokenizer Contex length: 512


## Choose an embedding Model

In [27]:
sentence_pairs = [
    ("What is the capital of France?", "Paris is the capital."),
    ("What is the capital of France?", "The Eiffel Tower is in Paris."),
    ("What is the capital of France?", "Berlin is a city in Germany.")
]

In [28]:
# 1. Load the Cross-Encoder model
# This model will predict a score (0 to 1) for the semantic similarity of the pair.
reranker_model = CrossEncoder(local_path + 'cross-encoder_stsb-roberta-large').to(DEVICE)

# 2. Score the pairs
# The .predict() method processes the pairs and returns a list of scores.
similarity_scores = reranker_model.predict(sentence_pairs)

print(f"Tokenizer Contex length: {reranker_model.max_length}")
similarity_scores

Tokenizer Contex length: 512


array([0.3123381 , 0.13270384, 0.01069016], dtype=float32)

In [30]:
# Load a cross-encoder that directly outputs similarity scores
reranker_model = CrossEncoder(local_path + 'cross-encoder_stsb-roberta-base').to(DEVICE)

# Predict similarity score
similarity = reranker_model.predict(sentence_pairs)

print(f"Tokenizer Contex length: {reranker_model.max_length}")

similarity

Tokenizer Contex length: 512


array([0.26188636, 0.21285567, 0.1247337 ], dtype=float32)

In [31]:
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# import torch
# import numpy as np

# global device and model initialization for performance
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# local path to the models
path = "/users/sir/downloads/HuggingFace/sentence_transformer/"

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
    # "She enjoys reading books on weekends.",
    # "The cat is sleeping on the couch."
]

# test multiple models
models = ['all-MiniLM-L6-v2','nli-mpnet-base-v2', 'all-mpnet-base-v2', 'all-mpnet-base-v2', 
          'paraphrase-mpnet-base-v2','BAAI/bge-base-en-v1.5', 'BAAI/bge-base-en-v1.5',
          'intfloat/e5-large-v2', 'BAAI/bge-large-en', 'allenai/scibert_scivocab_uncased',
          'thenlper/gte-large', 'mixedbread-ai/mxbai-embed-large-v1', # 'intfloat/e5-mistral-7b-instruct',
          'sentence-t5-large', 'thenlper/gte-large',
          'all-distilroberta-v1', 'roberta-large-nli-stsb-mean-tokens']



# uterate through models
for model_name in models:
    print(f"\nEvaluating model: {model_name}")
    # 1. Load a pretrained Sentence Transformer model
    model = SentenceTransformer(path+model_name.replace('/', '_')).to(DEVICE)

    # 2. Calculate embeddings by calling model.encode()
    embeddings = model.encode(sentences, normalize_embeddings=True)
    print(f"Embeddings shape for {model_name}: {embeddings.shape}")
    # display contex length
    print(f"{model_name} Max Length: {model.max_seq_length}")
    
    # 3. Compute cosine similarity between the first and second sentence embeddings
    similarity = cosine_similarity(embeddings)
    # view full similarity matrix & no scientific notation
    np.set_printoptions(precision=5, suppress=True)
    print(f"Similarity for {model_name}: \n{similarity}")


Evaluating model: all-MiniLM-L6-v2
Embeddings shape for all-MiniLM-L6-v2: (3, 384)
all-MiniLM-L6-v2 Max Length: 256
Similarity for all-MiniLM-L6-v2: 
[[1.      0.66596 0.10458]
 [0.66596 1.      0.14114]
 [0.10458 0.14114 1.     ]]

Evaluating model: nli-mpnet-base-v2
Embeddings shape for nli-mpnet-base-v2: (3, 768)
nli-mpnet-base-v2 Max Length: 75
Similarity for nli-mpnet-base-v2: 
[[1.      0.73911 0.00044]
 [0.73911 1.      0.02483]
 [0.00044 0.02483 1.     ]]

Evaluating model: all-mpnet-base-v2
Embeddings shape for all-mpnet-base-v2: (3, 768)
all-mpnet-base-v2 Max Length: 384
Similarity for all-mpnet-base-v2: 
[[1.      0.68165 0.0492 ]
 [0.68165 1.      0.04209]
 [0.0492  0.04209 1.     ]]

Evaluating model: all-mpnet-base-v2
Embeddings shape for all-mpnet-base-v2: (3, 768)
all-mpnet-base-v2 Max Length: 384
Similarity for all-mpnet-base-v2: 
[[1.      0.68165 0.0492 ]
 [0.68165 1.      0.04209]
 [0.0492  0.04209 1.     ]]

Evaluating model: paraphrase-mpnet-base-v2
Embeddings sh